# Train the semantic whole-latent scheduler

**Experiment boundary:** 60 distinct LADI v2 images, 36/12/12 group-safe
split, five simulated UEs, whole SwinJSCC latent jobs. P2 and P3 are
image-derived; P5, P6 and P7 require reviewed task annotations.
The DQN learns a *next-job decision*, not a one-shot permutation.

This notebook starts with a disposable clean clone of the latest remote
`semantic-aware-ai-scheduler` commit. Results go to a separate persistent
run directory. It refuses an unreviewed manifest or missing measured
payload when measured mode is selected.

## 1. Clean reset and exact source revision

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/saltypal/SemanticSchedulerEnd2End.git"
BRANCH = "semantic-aware-ai-scheduler"
IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB = "COLAB_RELEASE_TAG" in os.environ
WORKSPACE = Path("/kaggle/working" if IS_KAGGLE else os.environ.get("SCHEDULER_WORKSPACE", Path.cwd())).resolve()
WORKSPACE.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT = WORKSPACE / "project-src"
if PROJECT_ROOT.exists():
    if PROJECT_ROOT.resolve().parent != WORKSPACE:
        raise RuntimeError("Refusing to remove a checkout outside the disposable workspace")
    shutil.rmtree(PROJECT_ROOT)
subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)
subprocess.run(["git", "fetch", "origin", BRANCH], cwd=PROJECT_ROOT, check=True)
subprocess.run(["git", "checkout", "--detach", f"origin/{BRANCH}"], cwd=PROJECT_ROOT, check=True)
SOURCE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
sys.path.insert(0, str(PROJECT_ROOT))
print({"environment": "kaggle" if IS_KAGGLE else "colab" if IS_COLAB else "local",
       "branch": BRANCH, "exact_commit": SOURCE_COMMIT, "checkout": str(PROJECT_ROOT)})

## 2. Dependencies, hardware, and persistent output

In [ ]:
import importlib.metadata as package_metadata
import json, platform
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(PROJECT_ROOT / "environment/requirements-semantic-scheduler-kaggle.txt")], check=True)
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
if os.environ.get("TPU_NAME") or os.environ.get("XRT_TPU_CONFIG"):
    raise RuntimeError("This PyTorch DQN notebook has no TPU/XLA execution path; select CPU or CUDA")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ENVIRONMENT = {
    "python": platform.python_version(), "platform": platform.platform(),
    "device": DEVICE, "gpu_count": torch.cuda.device_count(),
    "gpu_name": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
    "packages": {name: package_metadata.version(name) for name in
                 ("torch", "numpy", "pandas", "scikit-learn", "transformers", "datasets")},
}
print(ENVIRONMENT)

In [ ]:
from datetime import datetime, timezone
from hashlib import sha256
from dataclasses import asdict
import yaml
from INFRA.ladi_manifest import validate_ladi60_manifest, render_contact_sheet
from INFRA.semantic_config import load_scheduler_config, load_ue_profiles
from INFRA.experiments.semantic_scheduler_experiment import (
    TASK_ID, prepare_feature_cache, load_catalog, build_suites_from_manifest,
    run_provenance, simulated_fixed_payload_costs,
)
from INFRA.semantic_contracts import ResourceBudgetSemantic
from INFRA.learning.supervised_training import fit_supervised_models
from INFRA.learning.dqn_training import DQNConfig, train_dqn_seeds
from EncDec.integration.scheduler_replay import cache_swinjscc_payloads

RESULTS_ROOT = Path(os.environ.get("SCHEDULER_RESULTS_DIR",
    "/kaggle/working/outputs" if IS_KAGGLE else WORKSPACE / "semantic-scheduler-results"))
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + SOURCE_COMMIT[:8]
RUN_ROOT = RESULTS_ROOT / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=False)
MANIFEST_PATH = Path(os.environ.get("SCHEDULER_REVIEWED_MANIFEST",
    "/kaggle/input/semantic-scheduler-ladi60-reviewed/ladi60_semantic_scheduler_v1.csv"))
PAYLOAD_MODE = os.environ.get("SCHEDULER_PAYLOAD_MODE", "measured")
if PAYLOAD_MODE not in ("measured", "simulated_fixed"):
    raise ValueError("SCHEDULER_PAYLOAD_MODE must be measured or simulated_fixed")
MODEL_DIR = Path(os.environ.get("SCHEDULER_SWIN_MODEL_DIR", PROJECT_ROOT / "models/SwinJSCC"))
UPSTREAM_ROOT = PROJECT_ROOT / "external/SwinJSCC"
SCHEDULER_SETTINGS = load_scheduler_config(PROJECT_ROOT / "INFRA/config/semantic_scheduler_demo.yaml")
UE_PROFILES = load_ue_profiles(PROJECT_ROOT / "INFRA/config/ue_profiles.yaml")
DQN_SETTINGS = yaml.safe_load((PROJECT_ROOT / "Scheduler/config/semantic_dqn.yaml").read_text(encoding="utf-8"))
if DQN_SETTINGS["episodes"] != SCHEDULER_SETTINGS.train_episodes_per_seed:
    raise ValueError("DQN and stream episode counts disagree")
(RUN_ROOT / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2), encoding="utf-8")
print({"run_id": RUN_ID, "output": str(RUN_ROOT), "payload_mode": PAYLOAD_MODE})

## 3. Reviewed dataset gate

Attach the private reviewed-image dataset and set
`SCHEDULER_REVIEWED_MANIFEST` to its CSV path if needed. The manifest
must contain 60 unique images, six selection strata, 30 reviewed
evidence pairs, and a 36/12/12 group-safe split. The draft contact
sheet is a review aid, never automatic evidence-group truth.

In [ ]:
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Reviewed manifest missing: {MANIFEST_PATH}. Attach the private reviewed LADI dataset "+
        "or set SCHEDULER_REVIEWED_MANIFEST before running this notebook.")
reviewed_rows = validate_ladi60_manifest(MANIFEST_PATH, require_reviewed=True)
frame = pd.DataFrame(reviewed_rows)
display(frame.groupby(["split", "selection_stratum"]).size().unstack(fill_value=0))
sheet = render_contact_sheet(MANIFEST_PATH, RUN_ROOT / "ladi60_contact_sheet.png")
from PIL import Image
display(Image.open(sheet))
print("All 60 image hashes, annotations and evidence groups validated.")

## 4. Extract CLIP features and cached image-classifier predictions once

In [ ]:
cache = prepare_feature_cache(MANIFEST_PATH, RUN_ROOT / "cache",
    include_ladi_predictions=True, device=DEVICE)
feature_rows = [json.loads(line) for line in cache["features"].read_text(encoding="utf-8").splitlines()]
extraction = pd.DataFrame({"image_id": row["image_id"],
                           "latency_ms": row["extraction_latency_ms"]} for row in feature_rows)
display(extraction.describe(include="all"))
print("Cached image embeddings are internal to INFRA; policies see five values and masks only.")

## 5. Cache payload cost once

Measured mode runs the existing checkpoint and records complex channel
uses separately from latent storage bytes. The checkpoint demands its
exported Python/PyTorch runtime; an incompatibility stops this cell.
`simulated_fixed` is only for an explicitly labelled infrastructure smoke run.

In [ ]:
if PAYLOAD_MODE == "simulated_fixed":
    PAYLOAD_COSTS = simulated_fixed_payload_costs(
        MANIFEST_PATH, RUN_ROOT / "payload" / "payload_costs.csv")
else:
    supplied_costs = os.environ.get("SCHEDULER_PAYLOAD_COSTS_CSV")
    if not supplied_costs:
        attached_costs = Path("/kaggle/input/semantic-scheduler-measured-costs/payload_costs.csv")
        supplied_costs = str(attached_costs) if attached_costs.is_file() else None
    if supplied_costs:
        PAYLOAD_COSTS = Path(supplied_costs)
        if not PAYLOAD_COSTS.is_file():
            raise FileNotFoundError(PAYLOAD_COSTS)
    else:
        PAYLOAD_COSTS = cache_swinjscc_payloads(
            reviewed_rows, RUN_ROOT / "payload", model_dir=MODEL_DIR,
            upstream_root=UPSTREAM_ROOT, device=DEVICE)
cost_frame = pd.read_csv(PAYLOAD_COSTS)
if cost_frame.resource_unit.nunique() != 1:
    raise ValueError("A run cannot mix resource units")
if PAYLOAD_MODE == "measured" and cost_frame.resource_unit.iloc[0] != "complex_channel_uses":
    raise ValueError("Measured mode requires checkpoint-derived channel uses")
display(cost_frame.head())

## 6. Freeze streams, UE profiles, deadlines, and transport tapes

In [ ]:
catalogue = load_catalog(MANIFEST_PATH, cache["features"], PAYLOAD_COSTS)
suites = build_suites_from_manifest(catalogue, MANIFEST_PATH,
                                     config=SCHEDULER_SETTINGS,
                                     ue_profiles=UE_PROFILES)
unit = catalogue[0].candidate.resource_unit
service_rate = max(item.candidate.resource_units for item in catalogue) / 10.0
budget = ResourceBudgetSemantic(service_rate, resource_unit=unit)
print({"train_episodes_per_seed": 300, "validation_episodes": len(suites["validation"]),
       "jobs_per_train_episode": len(suites["train_by_seed"][42][0].jobs),
       "service_rate_units_per_ms": service_rate, "resource_unit": unit})
episode_file = RUN_ROOT / "train_episodes.jsonl"
with episode_file.open("w", encoding="utf-8") as handle:
    for seed, episodes in suites["train_by_seed"].items():
        for episode in episodes:
            compact = {"episode_id": episode.episode_id, "seed": episode.seed,
                       "jobs": [{"job_id": job.candidate.job_id,
                                 "image_id": job.candidate.image_id,
                                 "ue_id": job.candidate.ue_id,
                                 "arrival_ms": job.candidate.arrival_ms,
                                 "deadline_ms": job.candidate.deadline_ms,
                                 "transport_success": job.transport_success}
                                for job in episode.jobs]}
            handle.write(json.dumps(compact) + "\n")

## 7. Fit the two supervised reference policies on 36 training images

In [ ]:
split_by_image = {row["image_id"]: row["split"] for row in reviewed_rows}
train_jobs = [job for job in catalogue if split_by_image[job.candidate.image_id] == "train"]
resolved_config = {
    "scheduler": asdict(SCHEDULER_SETTINGS), "ue_profiles": [asdict(p) for p in UE_PROFILES],
    "dqn": DQN_SETTINGS, "payload_mode": PAYLOAD_MODE,
    "service_rate_units_per_ms": service_rate, "resource_unit": unit,
    "seeds": [42, 43, 44],
}
(RUN_ROOT / "resolved_config.yaml").write_text(
    yaml.safe_dump(resolved_config, sort_keys=True), encoding="utf-8")
measured_checkpoint_sha = (str(cost_frame.checkpoint_sha256.iloc[0])
    if PAYLOAD_MODE == "measured" else None)
provenance = run_provenance(PROJECT_ROOT, MANIFEST_PATH,
    config=resolved_config, checkpoint_sha256=measured_checkpoint_sha)
if provenance["git_commit"] != SOURCE_COMMIT or provenance["git_dirty"]:
    raise RuntimeError("Training checkout changed after the clean Git sync")
models_dir = RUN_ROOT / "models"
supervised = fit_supervised_models(train_jobs, models_dir, provenance=provenance)
print({name: str(path) for name, path in supervised.items()})

## 8. Train three small standard DQNs

Each action selects one eligible whole latent. Replay uses a target
network, masked Bellman targets, Huber loss and 459→64→32→25 layers.
Validation uses fixed streams and normalized semantic-utility AUC.

In [ ]:
dqn_config = DQNConfig(**DQN_SETTINGS)
dqn_manifests = train_dqn_seeds(
    suites["train_by_seed"], suites["validation"], budget, models_dir,
    config=dqn_config, provenance=provenance, device=DEVICE)
training_rows = []
for seed in (42, 43, 44):
    path = models_dir / f"dqn_seed{seed}_history.json"
    training_rows.extend(json.loads(path.read_text(encoding="utf-8")))
training_frame = pd.DataFrame(training_rows)
training_frame.to_csv(RUN_ROOT / "training_metrics.csv", index=False)
display(training_frame.groupby("seed").tail(1))

## 9. Diagnostics and durable artifact manifest

In [ ]:
plots = RUN_ROOT / "plots"
plots.mkdir(exist_ok=True)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for seed, group in training_frame.groupby("seed"):
    axes[0].plot(group.episode, group.reward, label=str(seed), alpha=.8)
    axes[1].plot(group.episode, group.loss, label=str(seed), alpha=.8)
    observed = group.dropna(subset=["validation_utility_auc"])
    axes[2].plot(observed.episode, observed.validation_utility_auc, marker="o", label=str(seed))
for axis, title in zip(axes, ("Episode reward", "Huber loss", "Validation utility AUC")):
    axis.set_title(title); axis.set_xlabel("Episode"); axis.grid(alpha=.2)
axes[0].legend(title="Seed")
fig.tight_layout(); fig.savefig(plots / "training_diagnostics.png", dpi=170)
display(fig); plt.close(fig)
action_counts = np.sum([json.loads((models_dir / f"dqn_seed{seed}_actions.json")
                        .read_text(encoding="utf-8"))["action_index_counts"]
                        for seed in (42, 43, 44)], axis=0)
fig, axis = plt.subplots(figsize=(10, 3))
axis.bar(np.arange(25), action_counts, color="#13678A")
axis.set(xlabel="Eligible candidate index at decision", ylabel="Selections",
         title="DQN action-index distribution across three seeds")
fig.tight_layout(); fig.savefig(plots / "action_distribution.png", dpi=170)
display(fig); plt.close(fig)

import shutil
shutil.copy2(MANIFEST_PATH, RUN_ROOT / "dataset_manifest.csv")
shutil.copy2(PAYLOAD_COSTS, RUN_ROOT / "payload_costs.csv")
shutil.copy2(cache["features"], RUN_ROOT / "feature_cache.jsonl")
shutil.copy2(cache["predictions"], RUN_ROOT / "ladi_predictions.json")
def file_hash(path):
    return sha256(Path(path).read_bytes()).hexdigest()
required = ["resolved_config.yaml", "dataset_manifest.csv", "payload_costs.csv", "feature_cache.jsonl",
            "ladi_predictions.json", "models/ordinal_classifier.npz",
            "models/pairwise_ranker.npz"] + [f"models/dqn_seed{seed}_best.npz" for seed in (42, 43, 44)]
artifact_hashes = {name: file_hash(RUN_ROOT / name) for name in required}
run_manifest = {
    **provenance, "run_id": RUN_ID, "status": "trained_on_reviewed_manifest",
    "payload_mode": PAYLOAD_MODE, "resource_unit": unit,
    "service_rate_units_per_ms": service_rate,
    "artifacts_sha256": artifact_hashes,
    "dqn_best_validation": {str(seed): data["best_validation_utility_auc"]
                             for seed, data in dqn_manifests.items()},
}
(RUN_ROOT / "run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2, sort_keys=True), encoding="utf-8")
print("Persistent run:", RUN_ROOT)
print("Kaggle CLI: kaggle kernels output <username>/semantic-scheduler-train -p downloaded/train-run")